# Review Sentiment Trend Analysis

## Project Overview
Analyze customer review text and ratings over time to identify sentiment patterns, top complaints, top praised features, and changes in customer perception. This is a data analysis project using basic text analysis techniques — no ML model training.

## Learning Objectives
- Analyze rating distributions and trends over time
- Extract sentiment from review text using simple keyword-based approaches
- Identify top complaint themes and praised features
- Visualize sentiment evolution and detect perception shifts

## Problem Statement
A product team needs to understand how customer sentiment is evolving. Are reviews getting better or worse? What do customers praise? What do they complain about? Are there inflection points in perception?

## Why This Project Matters
Review sentiment analysis closes the feedback loop between customers and product teams. Detecting a sentiment drop early can prevent churn. Understanding what customers love helps prioritize features to keep and promote.

## Dataset Overview
Synthetic product review data with ~5K reviews over 2 years. Each review has a rating (1-5), text, product category, and date. The data includes injected sentiment shifts to simulate real-world events.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by common e-commerce review dynamics
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))

# Keyword dictionaries for simple sentiment analysis
POSITIVE_WORDS = {'great','excellent','love','amazing','perfect','awesome','fantastic',
                  'wonderful','best','happy','quality','recommend','easy','fast','beautiful',
                  'comfortable','durable','reliable','solid','impressed','smooth','intuitive'}
NEGATIVE_WORDS = {'bad','terrible','worst','hate','awful','poor','broken','defective',
                  'disappointed','slow','cheap','flimsy','useless','frustrating','waste',
                  'horrible','uncomfortable','unreliable','confusing','difficult','ugly','returned'}
print('Configuration set')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_reviews = 5000

categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Beauty', 'Sports']
cat_weights = [0.30, 0.25, 0.20, 0.15, 0.10]

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

# Positive and negative review fragments
pos_fragments = [
    'Great quality product', 'Exactly as described', 'Love this item',
    'Fast shipping and easy setup', 'Excellent value for money',
    'Works perfectly', 'Very comfortable and durable', 'Highly recommend',
    'Best purchase this year', 'Amazing build quality',
    'Smooth and intuitive to use', 'Beautiful design', 'Solid construction',
    'Impressed with the quality', 'Easy to use right out of the box',
]
neg_fragments = [
    'Poor quality material', 'Broke after a week', 'Not as described',
    'Very disappointed', 'Waste of money', 'Terrible customer service',
    'Cheap and flimsy', 'Returned immediately', 'Frustrating experience',
    'Defective on arrival', 'Uncomfortable and poorly made',
    'Confusing instructions', 'Slow and unreliable', 'Worst purchase ever',
    'Does not match the photos', 'Difficult to assemble',
]
neutral_fragments = [
    'It is okay for the price', 'Average product nothing special',
    'Does the job', 'Decent quality', 'Met expectations',
    'Not bad but not great either', 'Acceptable for daily use',
]

def gen_review(rating):
    if rating >= 4:
        parts = np.random.choice(pos_fragments, size=np.random.randint(1,3), replace=False)
    elif rating <= 2:
        parts = np.random.choice(neg_fragments, size=np.random.randint(1,3), replace=False)
    else:
        parts = np.random.choice(neutral_fragments, size=np.random.randint(1,2), replace=False)
    return '. '.join(parts) + '.'

# Generate with a sentiment shift: ratings drop mid-2024 for Electronics
rows = []
for i in range(n_reviews):
    cat = np.random.choice(categories, p=cat_weights)
    date = np.random.choice(dates)

    # Base rating distribution
    if cat == 'Electronics' and date >= pd.Timestamp('2024-07-01'):
        # Simulate quality issue event
        rating = np.random.choice([1,2,3,4,5], p=[0.20, 0.25, 0.25, 0.20, 0.10])
    elif cat == 'Beauty' and date >= pd.Timestamp('2024-04-01'):
        # Simulate product improvement
        rating = np.random.choice([1,2,3,4,5], p=[0.05, 0.05, 0.15, 0.35, 0.40])
    else:
        rating = np.random.choice([1,2,3,4,5], p=[0.08, 0.10, 0.20, 0.35, 0.27])

    text = gen_review(rating)
    rows.append({'ReviewID': i, 'Date': date, 'Category': cat, 'Rating': rating, 'ReviewText': text})

df = pd.DataFrame(rows)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df['YearMonth'] = df['Date'].dt.to_period('M')
print(f'Dataset shape: {df.shape}')
print(f'Average rating: {df["Rating"].mean():.2f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nRating distribution:\n{df["Rating"].value_counts().sort_index()}')
print(f'\nEmpty review texts: {(df["ReviewText"].str.strip() == "").sum()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Categories: {df["Category"].unique().tolist()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rating distribution
df['Rating'].value_counts().sort_index().plot.bar(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Rating Distribution')
axes[0,0].set_xlabel('Rating')

# Avg rating by category
df.groupby('Category')['Rating'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('Average Rating by Category')
axes[0,1].set_xlabel('Average Rating')

# Monthly avg rating
monthly_avg = df.groupby('YearMonth')['Rating'].mean()
monthly_avg.plot(ax=axes[1,0], marker='o')
axes[1,0].set_title('Monthly Average Rating')
axes[1,0].tick_params(axis='x', rotation=45)

# Review volume over time
monthly_count = df.groupby('YearMonth').size()
monthly_count.plot.bar(ax=axes[1,1], edgecolor='black', alpha=0.7)
axes[1,1].set_title('Monthly Review Volume')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Keyword-Based Sentiment Scoring

In [ ]:
def sentiment_score(text):
    words = set(re.findall(r'\b[a-z]+\b', text.lower()))
    pos = len(words & POSITIVE_WORDS)
    neg = len(words & NEGATIVE_WORDS)
    if pos > neg:
        return 'Positive'
    elif neg > pos:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['ReviewText'].apply(sentiment_score)
print(f'Sentiment distribution:\n{df["Sentiment"].value_counts()}')

# Validate: sentiment vs rating
sent_vs_rating = df.groupby('Sentiment')['Rating'].mean().round(2)
print(f'\nAverage rating by sentiment label:\n{sent_vs_rating}')

## Sentiment Trends Over Time

In [ ]:
sent_monthly = df.groupby(['YearMonth', 'Sentiment']).size().unstack(fill_value=0)
sent_monthly_pct = sent_monthly.div(sent_monthly.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sent_monthly_pct.plot(ax=axes[0], marker='o')
axes[0].set_title('Sentiment Distribution Over Time (%)')
axes[0].set_ylabel('Proportion')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Sentiment')

# Average rating trend by category
for cat in categories:
    cat_monthly = df[df['Category'] == cat].groupby('YearMonth')['Rating'].mean()
    cat_monthly.plot(ax=axes[1], label=cat, marker='o', alpha=0.7)
axes[1].set_title('Average Rating Trend by Category')
axes[1].set_ylabel('Avg Rating')
axes[1].legend(title='Category', bbox_to_anchor=(1.05, 1))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sentiment_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Top Complaints (Negative Reviews)

In [ ]:
neg_reviews = df[df['Rating'] <= 2]
print(f'Negative reviews (1-2 stars): {len(neg_reviews)}')

# Word frequency in negative reviews
neg_words_all = []
for text in neg_reviews['ReviewText']:
    words = re.findall(r'\b[a-z]+\b', text.lower())
    neg_words_all.extend([w for w in words if len(w) > 3 and w not in {'this','that','with','from','have','been','were','very','does','just','also'}])

neg_word_freq = Counter(neg_words_all).most_common(15)
print('\nTop words in negative reviews:')
for word, count in neg_word_freq:
    print(f'  {word}: {count}')

fig, ax = plt.subplots(figsize=(10, 5))
words, counts = zip(*neg_word_freq)
ax.barh(words, counts, edgecolor='black', color='salmon')
ax.set_title('Most Common Words in Negative Reviews')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top_complaints.png'), dpi=100, bbox_inches='tight')
plt.show()

## Top Praised Features (Positive Reviews)

In [ ]:
pos_reviews = df[df['Rating'] >= 4]
print(f'Positive reviews (4-5 stars): {len(pos_reviews)}')

pos_words_all = []
for text in pos_reviews['ReviewText']:
    words = re.findall(r'\b[a-z]+\b', text.lower())
    pos_words_all.extend([w for w in words if len(w) > 3 and w not in {'this','that','with','from','have','been','were','very','does','just','also'}])

pos_word_freq = Counter(pos_words_all).most_common(15)
print('\nTop words in positive reviews:')
for word, count in pos_word_freq:
    print(f'  {word}: {count}')

fig, ax = plt.subplots(figsize=(10, 5))
words, counts = zip(*pos_word_freq)
ax.barh(words, counts, edgecolor='black', color='lightgreen')
ax.set_title('Most Common Words in Positive Reviews')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top_praised.png'), dpi=100, bbox_inches='tight')
plt.show()

## Detecting Sentiment Shifts

In [ ]:
# Electronics: expected drop starting July 2024
elec = df[df['Category'] == 'Electronics'].copy()
elec_monthly = elec.groupby('YearMonth')['Rating'].mean()

# Beauty: expected improvement starting April 2024
beauty = df[df['Category'] == 'Beauty'].copy()
beauty_monthly = beauty.groupby('YearMonth')['Rating'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
elec_monthly.plot(ax=axes[0], marker='o', color='red')
axes[0].axvline(x=elec_monthly.index.get_loc(pd.Period('2024-07', 'M')), color='grey', linestyle='--', label='Event: July 2024')
axes[0].set_title('Electronics — Rating Drop Event')
axes[0].set_ylabel('Avg Rating')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

beauty_monthly.plot(ax=axes[1], marker='o', color='green')
axes[1].axvline(x=beauty_monthly.index.get_loc(pd.Period('2024-04', 'M')), color='grey', linestyle='--', label='Event: April 2024')
axes[1].set_title('Beauty — Rating Improvement Event')
axes[1].set_ylabel('Avg Rating')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sentiment_shifts.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Electronics avg before Jul 2024:', elec[elec['Date'] < '2024-07-01']['Rating'].mean().round(2))
print('Electronics avg after Jul 2024:', elec[elec['Date'] >= '2024-07-01']['Rating'].mean().round(2))
print('Beauty avg before Apr 2024:', beauty[beauty['Date'] < '2024-04-01']['Rating'].mean().round(2))
print('Beauty avg after Apr 2024:', beauty[beauty['Date'] >= '2024-04-01']['Rating'].mean().round(2))

## Interpretation and Business Insight
- **Electronics** experienced a clear rating decline starting July 2024, suggesting a quality issue or supply chain problem
- **Beauty** saw improved ratings from April 2024 onward, possibly due to a product reformulation or packaging upgrade
- Top complaints center around "poor quality", "defective", and "disappointed" — indicating product durability issues
- Top praise focuses on "quality", "easy", "recommend", and "comfortable" — customers value reliability and convenience
- Keyword-based sentiment labels align well with numeric ratings, validating the simple approach for trend analysis

## Limitations
- Synthetic data with injected events — real sentiment is noisier and harder to interpret
- Keyword-based sentiment is simplistic; sarcasm, negation, and context are not handled
- No topic modeling (LDA, BERTopic) — would provide more nuanced theme extraction
- Review volume differences between categories may skew comparisons

## How to Improve This Project
- Use real product review data (e.g., Amazon Reviews dataset)
- Apply transformer-based sentiment analysis (e.g., VADER, TextBlob, or HuggingFace models)
- Add topic modeling to automatically discover complaint/praise themes
- Build an automated anomaly detection system for rating drops
- Segment analysis by customer tenure or purchase history

## Production Considerations
- Set up weekly sentiment dashboards by product category
- Configure alerts for sustained rating drops (e.g., 3+ weeks below category average)
- Feed top complaint themes into product development sprints
- Track sentiment before/after product changes to measure impact

## Common Mistakes
- Relying solely on average ratings (distribution shape matters more)
- Not accounting for review volume when interpreting trends (low-volume months are noisy)
- Treating all reviews equally regardless of helpfulness votes or reviewer credibility
- Ignoring sentiment context (e.g., "not bad" classified as negative due to "bad")

## Mini Challenge / Exercises
1. Add a "Review Length" column. Do longer reviews tend to be more negative?
2. Build a word cloud for each rating level (1-5).
3. Detect the month with the sharpest rating drop for any category.
4. Add bigram analysis (two-word phrases) to find themes like "customer service" or "easy setup".

## Final Summary / Key Takeaways
- Review sentiment analysis reveals both macro trends and specific complaint/praise themes
- Even simple keyword-based sentiment provides useful directional insights
- Detecting sentiment shifts early enables proactive quality interventions
- Combining ratings, text analysis, and temporal trends gives a complete picture of customer perception
- Business impact comes from acting on insights: fixing top complaints and promoting top-praised features